In [1]:
from pathlib import Path

import polars as pl
from IPython.display import display

In [2]:
metadata_path = Path("../../data/ISIC/metadata.csv")
metadata_df = pl.read_csv(metadata_path)
data_path = Path("../../data/ISIC")
image_path = data_path / "ISIC_2024_Training_Input"
ground_truth_path = data_path / "ISIC_2024_Training_GroundTruth.csv"
ground_truth_df = pl.read_csv(ground_truth_path)
merged_df = ground_truth_df.join(metadata_df, on="isic_id", how="inner")

total_merged = merged_df.height
malignant_merged = merged_df.filter(pl.col("malignant") == 1).height

print(f"Total images with metadata: {total_merged}")
print(f"Malignant images with metadata: {malignant_merged}")
print(f"Ratio: 1:{(total_merged - malignant_merged) / max(malignant_merged, 1):.1f}")

Total images with metadata: 401059
Malignant images with metadata: 393
Ratio: 1:1019.5


In [3]:
cols_to_check = [
    "age_approx",
    "sex",
    "anatom_site_general",
    "clin_size_long_diam_mm",
    "tbp_lv_areaMM2",
    "tbp_lv_eccentricity",
    "tbp_lv_nevi_confidence",
]

total_rows = merged_df.height
null_counts = merged_df.select([pl.col(c).is_null().sum().alias(c) for c in cols_to_check])

for col in cols_to_check:
    nulls = null_counts.item(0, col)
    print(f"{col}: {nulls} nulls ({(nulls / total_rows) * 100:.1f}% missing)")

age_approx: 2798 nulls (0.7% missing)
sex: 11517 nulls (2.9% missing)
anatom_site_general: 5756 nulls (1.4% missing)
clin_size_long_diam_mm: 0 nulls (0.0% missing)
tbp_lv_areaMM2: 0 nulls (0.0% missing)
tbp_lv_eccentricity: 0 nulls (0.0% missing)
tbp_lv_nevi_confidence: 0 nulls (0.0% missing)


In [4]:
patient_counts = merged_df.group_by("patient_id").agg(pl.count("isic_id").alias("num_images"))

print(f"Unique patients: {patient_counts.height}")
display(patient_counts["num_images"].describe())

print("\nTop 10 patients by image count:")
display(patient_counts.sort("num_images", descending=True).head(10))

Unique patients: 1042


statistic,value
str,f64
"""count""",1042.0
"""null_count""",0.0
"""mean""",384.893474
"""std""",540.268913
"""min""",1.0
"""25%""",115.0
"""50%""",242.0
"""75%""",478.0
"""max""",9184.0



Top 10 patients by image count:


patient_id,num_images
str,u32
"""IP_1117889""",9184
"""IP_5714646""",6267
"""IP_3921915""",5568
"""IP_7797815""",4454
"""IP_9577633""",3583
"""IP_5539318""",2859
"""IP_9853536""",2327
"""IP_5143034""",2283
"""IP_0321326""",2245


In [5]:
patient_mal_stats = merged_df.group_by("patient_id").agg(pl.col("malignant").max().alias("has_malignant"))

has_mal = patient_mal_stats.filter(pl.col("has_malignant") == 1).height
no_mal = patient_mal_stats.filter(pl.col("has_malignant") == 0).height

print(f"Patients with >= 1 malignant: {has_mal}")
print(f"Patients with 0 malignant: {no_mal}")

Patients with >= 1 malignant: 259
Patients with 0 malignant: 783


In [6]:
supplement_path = Path("../../data/ISIC/ISIC_2024_Training_Supplement.csv")
supplement_df = pl.read_csv(supplement_path)

total_supp = supplement_df.height
null_counts_supp = supplement_df.select([pl.col(c).is_null().sum().alias(c) for c in supplement_df.columns])

for col in supplement_df.columns:
    nulls = null_counts_supp.item(0, col)
    print(f"{col}: {nulls} nulls ({(nulls / total_supp) * 100:.1f}% missing)")

isic_id: 0 nulls (0.0% missing)
attribution: 0 nulls (0.0% missing)
copyright_license: 0 nulls (0.0% missing)
lesion_id: 379001 nulls (94.5% missing)
iddx_full: 0 nulls (0.0% missing)
iddx_1: 0 nulls (0.0% missing)
iddx_2: 399991 nulls (99.7% missing)
iddx_3: 399994 nulls (99.7% missing)
iddx_4: 400508 nulls (99.9% missing)
iddx_5: 401058 nulls (100.0% missing)
mel_mitotic_index: 401006 nulls (100.0% missing)
mel_thick_mm: 400996 nulls (100.0% missing)
tbp_lv_dnn_lesion_confidence: 0 nulls (0.0% missing)


In [7]:
cat_cols = ["sex", "anatom_site_general"]
for col in cat_cols:
    print(f"--- {col} ---")
    display(
        merged_df.group_by(col)
        .agg(
            pl.count("isic_id").alias("total"),
            pl.col("malignant").sum().alias("malignant"),
            (pl.col("malignant").sum() / pl.count("isic_id")).alias("mal_rate"),
        )
        .sort("total", descending=True)
    )
    print()

num_cols = ["age_approx", "clin_size_long_diam_mm", "tbp_lv_areaMM2", "tbp_lv_eccentricity"]
exprs = []
for c in num_cols:
    exprs.append(pl.col(c).median().alias(f"{c}_median"))
    exprs.append(pl.col(c).mean().alias(f"{c}_mean"))
    exprs.append(pl.col(c).min().alias(f"{c}_min"))
    exprs.append(pl.col(c).max().alias(f"{c}_max"))

print("--- Numerical Features ---")
display(merged_df.group_by("malignant").agg(exprs))

--- sex ---


sex,total,malignant,mal_rate
str,u32,f64,f64
"""male""",265546,274.0,0.001032
"""female""",123996,109.0,0.000879
null,11517,10.0,0.000868



--- anatom_site_general ---


anatom_site_general,total,malignant,mal_rate
str,u32,f64,f64
"""posterior torso""",121902,103.0,0.000845
"""lower extremity""",103028,73.0,0.000709
"""anterior torso""",87770,82.0,0.000934
"""upper extremity""",70557,57.0,0.000808
"""head/neck""",12046,78.0,0.006475
null,5756,0.0,0.0



--- Numerical Features ---


malignant,age_approx_median,age_approx_mean,age_approx_min,age_approx_max,clin_size_long_diam_mm_median,clin_size_long_diam_mm_mean,clin_size_long_diam_mm_min,clin_size_long_diam_mm_max,tbp_lv_areaMM2_median,tbp_lv_areaMM2_mean,tbp_lv_areaMM2_min,tbp_lv_areaMM2_max,tbp_lv_eccentricity_median,tbp_lv_eccentricity_mean,tbp_lv_eccentricity_min,tbp_lv_eccentricity_max
f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,60.0,58.009694,5,85,3.37,3.929043,1.0,28.4,5.68587,8.526291,0.431601,334.1527,0.768237,0.741262,0.027667,0.9749603
1.0,60.0,61.371795,20,85,5.14,5.749771,1.01,18.94,12.85419,22.490554,0.6567836,141.114656,0.738177,0.716914,0.193099,0.960627


In [8]:
size_cols = ["clin_size_long_diam_mm", "tbp_lv_areaMM2"]
for c in size_cols:
    print(f"--- {c} ---")
    display(
        merged_df.group_by("malignant").agg(
            [
                pl.col(c).median().alias("median"),
                pl.col(c).mean().alias("mean"),
                pl.col(c).min().alias("min"),
                pl.col(c).max().alias("max"),
            ]
        )
    )

--- clin_size_long_diam_mm ---


malignant,median,mean,min,max
f64,f64,f64,f64,f64
0.0,3.37,3.929043,1.0,28.4
1.0,5.14,5.749771,1.01,18.94


--- tbp_lv_areaMM2 ---


malignant,median,mean,min,max
f64,f64,f64,f64,f64
0.0,5.68587,8.526291,0.431601,334.1527
1.0,12.85419,22.490554,0.6567836,141.114656


In [9]:
caps = [10, 20, 50]
total_malignants = merged_df.filter(pl.col("malignant") == 1).height

patient_has_mal = merged_df.group_by("patient_id").agg(pl.col("malignant").max().alias("has_mal"))
df_with_flag = merged_df.join(patient_has_mal, on="patient_id")

benign_from_benign_only = df_with_flag.filter((pl.col("malignant") == 0) & (pl.col("has_mal") == 0))
benign_from_mixed = df_with_flag.filter((pl.col("malignant") == 0) & (pl.col("has_mal") == 1))

print(f"Baseline benigns from benign-only patients: {benign_from_benign_only.height}")
print(f"Baseline benigns from patients WITH malignancy: {benign_from_mixed.height}")

for cap in caps:
    capped_benign_only = (
        benign_from_benign_only.with_columns(pl.int_range(pl.len()).over("patient_id").alias("row_num"))
        .filter(pl.col("row_num") < cap)
        .height
    )

    capped_all = (
        df_with_flag.filter(pl.col("malignant") == 0)
        .with_columns(pl.int_range(pl.len()).over("patient_id").alias("row_num"))
        .filter(pl.col("row_num") < cap)
        .height
    )

    total_scenario_1 = capped_benign_only + benign_from_mixed.height
    total_scenario_2 = capped_all

    print(f"\nCap = {cap} per patient:")
    print(
        f"  Capping ONLY benign-only patients: {total_scenario_1} benigns | "
        f"Total: {total_scenario_1 + total_malignants} | Ratio 1:{total_scenario_1 / total_malignants:.1f}"
    )
    print(
        f"  Capping ALL benigns:               {total_scenario_2} benigns | "
        f"Total: {total_scenario_2 + total_malignants} | Ratio 1:{total_scenario_2 / total_malignants:.1f}"
    )

Baseline benigns from benign-only patients: 245055
Baseline benigns from patients WITH malignancy: 155611

Cap = 10 per patient:
  Capping ONLY benign-only patients: 163349 benigns | Total: 163742 | Ratio 1:415.6
  Capping ALL benigns:               10310 benigns | Total: 10703 | Ratio 1:26.2

Cap = 20 per patient:
  Capping ONLY benign-only patients: 170911 benigns | Total: 171304 | Ratio 1:434.9
  Capping ALL benigns:               20442 benigns | Total: 20835 | Ratio 1:52.0

Cap = 50 per patient:
  Capping ONLY benign-only patients: 192396 benigns | Total: 192789 | Ratio 1:489.6
  Capping ALL benigns:               49626 benigns | Total: 50019 | Ratio 1:126.3


In [10]:
patient_mal_counts = (
    merged_df.filter(pl.col("malignant") == 1).group_by("patient_id").agg(pl.count("isic_id").alias("num_malignant"))
)

print("--- Malignant images per patient ---")
display(patient_mal_counts["num_malignant"].describe())

print("\nPatients with >1 malignant image:")
display(patient_mal_counts.filter(pl.col("num_malignant") > 1).sort("num_malignant", descending=True))

--- Malignant images per patient ---


statistic,value
str,f64
"""count""",259.0
"""null_count""",0.0
"""mean""",1.517375
"""std""",1.303947
"""min""",1.0
"""25%""",1.0
"""50%""",1.0
"""75%""",2.0
"""max""",14.0



Patients with >1 malignant image:


patient_id,num_malignant
str,u32
"""IP_2456971""",14
"""IP_1959239""",8
"""IP_0669361""",7
"""IP_9324599""",7
"""IP_3905195""",6
…,…
"""IP_2331257""",2
"""IP_0889220""",2
"""IP_9743101""",2
